In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from  sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [185]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
## preprocess the data
## drop irrelvant columns
data.drop(["RowNumber","CustomerId","Surname"],axis=1)

In [ ]:
##Encode categorical variable 
label_encoder_gender=LabelEncoder()
data["Gender"]=label_encoder_gender.fit_transform(data["Gender"])
data

In [186]:
## onehot encode "Geography"
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder(sparse_output=False)           # ← renamed param
geo_encoder = onehot_encoder_geo.fit_transform(data[["Geography"]])
geo_encoder_df = pd.DataFrame(geo_encoder,
                              columns=onehot_encoder_geo.get_feature_names_out(["Geography"]))
data = pd.concat([data.drop("Geography", axis=1), geo_encoder_df], axis=1)
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,1,15634602,Hargrave,619,Female,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,2,15647311,Hill,608,Female,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,3,15619304,Onio,502,Female,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,4,15701354,Boni,699,Female,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,5,15737888,Mitchell,850,Female,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [ ]:
with open('lable_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

In [ ]:
## Divide the dataset into dependent and independent features
X=data.drop(['Exited','RowNumber','CustomerId','Surname'],axis=1)
Y=data["Exited"]

#split the data in training and testing data
x_train ,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

##Scale these features
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)
x_train

In [ ]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

ANN Implementation

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [ ]:
## Build ANN Model

model=Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)), ##Hidden Layer 1 connected with input layer
    Dense(32,activation='relu'), #hidden layer -2
    Dense(1,activation='sigmoid') #output layer
]
)

In [ ]:
model.summary()

In [ ]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()
loss

In [ ]:
model.compile(optimizer=opt,loss="BinaryCrossentropy",metrics=["accuracy"])

In [ ]:
## Set up the Tensorboard
from  tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir="logs/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [ ]:
## Set up Early Stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [ ]:
##Train the mode
histroy=model.fit(
    x_train,y_train,validation_data=(x_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.save('model.h5')

In [ ]:
#Load Tensorboard Extension
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit